# Snowpark Notebook: SCD Type 2 Dimension Loads

This notebook applies SCD Type 2 logic from `STAGING` tables into `DW` dimension tables, maintaining `_valid_from`, `_valid_to`, and `_is_current`.

In [ ]:
import os
import pandas as pd
from typing import Optional
from snowflake.snowpark import Session


def _env(name: str, default: Optional[str] = None) -> str:
    value = os.getenv(name, default)
    if value is None:
        raise EnvironmentError(f"Missing required environment variable: {name}")
    return value


connection_parameters = {
    'account': _env('SNOWFLAKE_ACCOUNT'),
    'user': _env('SNOWFLAKE_USER'),
    'password': _env('SNOWFLAKE_PASSWORD'),
    'role': _env('SNOWFLAKE_ROLE', 'ACCOUNTADMIN'),
    'warehouse': _env('SNOWFLAKE_WAREHOUSE', 'ADVENTUREWORKS_ETL_WH'),
    'database': _env('SNOWFLAKE_DATABASE', 'ADVENTUREWORKS_MIGRATED'),
}

DATABASE = connection_parameters['database']
STAGING_SCHEMA = 'STAGING'
DW_SCHEMA = 'DW'

session = Session.builder.configs(connection_parameters).create()
session.sql('SELECT CURRENT_VERSION() AS snowflake_version').show()


In [ ]:
def quote_ident(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def fqtn(database: str, schema: str, table: str) -> str:
    return f"{quote_ident(database)}.{quote_ident(schema)}.{quote_ident(table)}"


def table_columns(schema: str, table: str) -> list[str]:
    sql = f"""
      SELECT COLUMN_NAME
      FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.COLUMNS
      WHERE TABLE_SCHEMA = '{schema.upper()}'
        AND TABLE_NAME = '{table.upper()}'
      ORDER BY ORDINAL_POSITION
    """
    return [row['COLUMN_NAME'] for row in session.sql(sql).collect()]


def hash_expr(alias: str, columns: list[str]) -> str:
    pairs = ', '.join([f"'{col}', {alias}.{quote_ident(col)}" for col in columns])
    return f"SHA2(TO_JSON(OBJECT_CONSTRUCT_KEEP_NULL({pairs})), 256)"


def business_key(columns: list[str]) -> Optional[str]:
    for col in columns:
        if col.upper().endswith('KEY'):
            return col
    return None


special_stage_tables = {
    'DIM_PROSPECTIVE_BUYER': 'PROSPECTIVE_BUYER_STG',
}

all_dw_tables = [
    row['TABLE_NAME']
    for row in session.sql(
        f"""
        SELECT TABLE_NAME
        FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{DW_SCHEMA}'
          AND TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME
        """
    ).collect()
]

mappings: list[dict] = []
for target_table in all_dw_tables:
    if not target_table.upper().startswith('DIM_'):
        continue

    source_table = special_stage_tables.get(target_table.upper(), f"{target_table}_STG")
    staging_exists = session.sql(
        f"""
        SELECT COUNT(*) AS CNT
        FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{STAGING_SCHEMA}'
          AND TABLE_NAME = '{source_table.upper()}'
        """
    ).collect()[0]['CNT']
    if staging_exists == 0:
        continue

    target_columns = table_columns(DW_SCHEMA, target_table)
    source_columns = set(table_columns(STAGING_SCHEMA, source_table))

    payload_columns = [
        c for c in target_columns
        if c.upper() not in {'_VALID_FROM', '_VALID_TO', '_IS_CURRENT'} and c in source_columns
    ]
    key_col = business_key(payload_columns)
    if key_col is None:
        continue

    mappings.append(
        {
            'source_table': source_table,
            'target_table': target_table,
            'business_key': key_col,
            'payload_columns': payload_columns,
        }
    )

pd.DataFrame(mappings)


In [ ]:
def apply_scd_type2(mapping: dict) -> dict:
    source_table = mapping['source_table']
    target_table = mapping['target_table']
    key_col = mapping['business_key']
    payload = mapping['payload_columns']

    source_fqn = fqtn(DATABASE, STAGING_SCHEMA, source_table)
    target_fqn = fqtn(DATABASE, DW_SCHEMA, target_table)
    delta_fqn = fqtn(DATABASE, DW_SCHEMA, f"TMP_{target_table}_DELTA")

    select_payload = ', '.join([f"s.{quote_ident(col)}" for col in payload])
    insert_columns = ', '.join([quote_ident(col) for col in payload] + ['_valid_from', '_valid_to', '_is_current'])
    insert_values = ', '.join([f"d.{quote_ident(col)}" for col in payload] + ['CURRENT_TIMESTAMP()', 'NULL', 'TRUE'])

    src_hash = hash_expr('s', payload)
    tgt_hash = hash_expr('t', payload)

    create_delta_sql = f"""
      CREATE OR REPLACE TEMP TABLE {delta_fqn} AS
      WITH source_data AS (
        SELECT {select_payload}, {src_hash} AS src_hash
        FROM {source_fqn} s
        QUALIFY ROW_NUMBER() OVER (
          PARTITION BY s.{quote_ident(key_col)}
          ORDER BY s.{quote_ident(key_col)}
        ) = 1
      ),
      current_data AS (
        SELECT {', '.join([f't.{quote_ident(col)}' for col in payload])}, {tgt_hash} AS tgt_hash
        FROM {target_fqn} t
        WHERE t._is_current = TRUE
      )
      SELECT s.*,
             CASE WHEN t.{quote_ident(key_col)} IS NULL THEN 'NEW' ELSE 'CHANGED' END AS change_type
      FROM source_data s
      LEFT JOIN current_data t
        ON s.{quote_ident(key_col)} = t.{quote_ident(key_col)}
      WHERE t.{quote_ident(key_col)} IS NULL
         OR s.src_hash <> t.tgt_hash
    """

    update_sql = f"""
      UPDATE {target_fqn} tgt
      SET _valid_to = CURRENT_TIMESTAMP(),
          _is_current = FALSE
      FROM {delta_fqn} d
      WHERE d.change_type = 'CHANGED'
        AND tgt.{quote_ident(key_col)} = d.{quote_ident(key_col)}
        AND tgt._is_current = TRUE
    """

    insert_sql = f"""
      INSERT INTO {target_fqn} ({insert_columns})
      SELECT {insert_values}
      FROM {delta_fqn} d
    """

    session.sql(create_delta_sql).collect()
    session.sql(update_sql).collect()
    session.sql(insert_sql).collect()

    delta_metrics = session.sql(
        f"""
        SELECT
          COUNT(*) AS TOTAL_ROWS,
          SUM(CASE WHEN change_type = 'NEW' THEN 1 ELSE 0 END) AS NEW_ROWS,
          SUM(CASE WHEN change_type = 'CHANGED' THEN 1 ELSE 0 END) AS CHANGED_ROWS
        FROM {delta_fqn}
        """
    ).collect()[0].as_dict()

    return {
        'source_table': source_table,
        'target_table': target_table,
        'business_key': key_col,
        **delta_metrics,
    }


run_results = [apply_scd_type2(mapping) for mapping in mappings]
pd.DataFrame(run_results)


In [ ]:
session.close()
print('SCD Type 2 notebook execution completed.')
